In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat_ws

spark = (SparkSession.builder
    .appName("F1_Preguntas")
    .enableHiveSupport()
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

spark.sql("USE f1_dw")

DataFrame[]

In [4]:
# Pregunta 3
q3 = spark.sql("""
SELECT nombre, apellido, carreras_validas, ganancia_promedio, pos
FROM (
    SELECT d.givenName  AS nombre,
           d.familyName AS apellido,
           COUNT(*)     AS carreras_validas,
           ROUND(AVG(f.grid - f.position), 2) AS ganancia_promedio,
           RANK() OVER (ORDER BY AVG(f.grid - f.position) DESC) AS pos
    FROM fact_results f
    JOIN dim_driver d ON f.driverId = d.driverId
    WHERE f.positionText RLIKE '^[0-9]+$'   -- solo clasificados (excluye R, D, W)
      AND f.grid > 0                        -- excluye largadas desde boxes
    GROUP BY d.givenName, d.familyName
    HAVING COUNT(*) >= 20 -- mínima cantidad de carreras para considerar piloto
) t
WHERE pos <= 10
ORDER BY pos, apellido, nombre;
""");

q3_anl = q3.withColumn("piloto", concat_ws(" ", "nombre", "apellido"))

In [5]:
q3_anl.printSchema()

root
 |-- nombre: string (nullable = true)
 |-- apellido: string (nullable = true)
 |-- carreras_validas: long (nullable = false)
 |-- ganancia_promedio: double (nullable = true)
 |-- pos: integer (nullable = false)
 |-- piloto: string (nullable = false)



In [6]:
q3_anl.coalesce(1).write.mode("overwrite").parquet("ort/ANL/pregunta3")